# FLAN-T5 SQuADv2 question-generation fine-tuning (Kaggle GPU)

Self-contained Kaggle version of `training/train_qg.py` with OOM fixes:
- Forces single-GPU usage (`CUDA_VISIBLE_DEVICES='0'`) to eliminate multi-GPU memory duplication
- Direct half-precision loading (`torch_dtype` in `from_pretrained`)
- Reduced batch size (`BATCH_SIZE=4`, `GRAD_ACCUM=4`)
- Periodic evaluation buffer flushing (`eval_accumulation_steps=1`)

**Before running:**
1. Restart kernel / session if previous run failed with OOM (`Run` -> `Restart Session`).
2. Attach your dataset containing `squadv2_train.jsonl` (Add Data, top right).
3. Enable GPU: Settings -> Accelerator -> GPU T4.
4. Run all cells top to bottom.

In [ ]:
import os
# Force single-GPU execution to prevent multi-GPU memory overhead on Kaggle T4
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

import glob
import json
import random
import shutil
from pathlib import Path

import torch

WORK = Path('/kaggle/working')
DATA_DIR = WORK / 'data' / 'training'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK / 'models' / 'question_generation' / 'flan-t5-squadv2'

# ---------------- training configuration (edit here) ----------------
MODEL_NAME = 'google/flan-t5-base'
EPOCHS = 2
BATCH_SIZE = 4             # Per device; reduced from 8 to prevent VRAM spikes
GRAD_ACCUM = 4             # Effective batch size = 4 x 4 = 16
GRADIENT_CHECKPOINTING = False  # Big VRAM saver for T5
MAX_TRAIN_SAMPLES = 40000  # None = use all ~86k examples
EVAL_GEN_SAMPLES = 500     # BLEU/ROUGE computed on this many held-out samples only
SEED = 42
# ---------------------------------------------------------------------

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
assert torch.cuda.is_available(), 'No GPU detected. Enable Settings -> Accelerator -> GPU T4.'

In [ ]:
# Locate uploaded dataset files anywhere under /kaggle/input
def find_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return Path(hits[0]) if hits else None

train_src = find_file('squadv2_train.jsonl')
val_src = find_file('squadv2_validation.jsonl')
assert train_src is not None, 'squadv2_train.jsonl not found. Attach your Kaggle dataset first.'
print('train source:', train_src)

train_path = DATA_DIR / 'squadv2_train.jsonl'
val_path = DATA_DIR / 'squadv2_validation.jsonl'

if val_src is not None:
    shutil.copy(train_src, train_path)
    shutil.copy(val_src, val_path)
    print('validation source:', val_src)
else:
    # No validation file uploaded: carve a held-out split from train (no overlap).
    lines = train_src.read_text(encoding='utf-8').splitlines()
    rng = random.Random(SEED)
    val_idx = set(rng.sample(range(len(lines)), 2000))
    with open(train_path, 'w', encoding='utf-8') as ft, open(val_path, 'w', encoding='utf-8') as fv:
        for i, ln in enumerate(lines):
            (fv if i in val_idx else ft).write(ln + '\n')
    print(f'Carved 2000 held-out validation records; train now {len(lines) - 2000} examples.')

print('train file:', train_path, '| val file:', val_path)
print('train lines:', sum(1 for _ in open(train_path, encoding='utf-8')))
print('val lines:', sum(1 for _ in open(val_path, encoding='utf-8')))

In [ ]:
# Metrics deps
!pip install -q rouge-score nltk

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    set_seed,
)

set_seed(SEED)
dataset = load_dataset('json', data_files={'train': str(train_path), 'validation': str(val_path)})
if MAX_TRAIN_SAMPLES and MAX_TRAIN_SAMPLES < len(dataset['train']):
    dataset['train'] = dataset['train'].select(range(MAX_TRAIN_SAMPLES))
print('train examples:', len(dataset['train']), '| validation examples:', len(dataset['validation']))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Determine GPU precision type
# T5 fine-tuning diverges in fp16 on T4-class GPUs (Turing). Use bf16 only when
# natively supported (Ampere+); otherwise train in fp32. Never use fp16 for T5.
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

# SDPA attention + load direct to fp16/bf16 to avoid FP32 memory spike
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        attn_implementation='sdpa'
    )
except (ValueError, TypeError):
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype
    )

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], max_length=512, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=64, truncation=True)
    
    # Replace pad_token_id (0) with -100 so PyTorch loss ignores padding positions
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    inputs['labels'] = label_ids
    return inputs

tokenized = dataset.map(
    tokenize, batched=True, num_proc=2,
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing question-generation data',
)

eval_gen = tokenized['validation']
if EVAL_GEN_SAMPLES and len(eval_gen) > EVAL_GEN_SAMPLES:
    eval_gen = eval_gen.shuffle(seed=SEED).select(range(EVAL_GEN_SAMPLES))
print('generation metrics computed on', len(eval_gen), 'samples')

In [ ]:
# BLEU / ROUGE / exact-match helpers
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer

def compute_generation_metrics(preds, refs):
    smooth = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    em = b1 = b2 = b4 = r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        if ' '.join(p.lower().split()) == ' '.join(r.lower().split()):
            em += 1
        rt, pt = [r.lower().split()], p.lower().split()
        if pt:
            b1 += sentence_bleu(rt, pt, weights=(1, 0, 0, 0), smoothing_function=smooth)
            b2 += sentence_bleu(rt, pt, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
            b4 += sentence_bleu(rt, pt, weights=(0.25,) * 4, smoothing_function=smooth)
        sc = scorer.score(r, p)
        r1 += sc['rouge1'].fmeasure; r2 += sc['rouge2'].fmeasure; rl += sc['rougeL'].fmeasure
    n = max(len(preds), 1)
    return {
        'exact_match': round(em / n, 4),
        'bleu1': round(b1 / n * 100, 2), 'bleu2': round(b2 / n * 100, 2), 'bleu4': round(b4 / n * 100, 2),
        'rouge1': round(r1 / n, 4), 'rouge2': round(r2 / n, 4), 'rougeL': round(rl / n, 4),
    }

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return compute_generation_metrics(decoded_preds, decoded_labels)

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print('mixed precision:', 'bf16' if bf16_ok else ('fp16' if torch.cuda.is_available() else 'off'))

OUT_DIR.mkdir(parents=True, exist_ok=True)
args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=bf16_ok,
    fp16=False,  # fp16 fine-tuning of T5 diverges; keep fp32/bf16 only
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    predict_with_generate=False,  # BLEU/ROUGE computed on eval_gen only
    generation_max_length=64,
    generation_num_beams=1,
    eval_accumulation_steps=1,  # Clears eval logits to CPU buffer
    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available(),
    optim='adamw_torch',
    save_only_model=True,
    report_to=[],
    load_best_model_at_end=True,
    save_total_limit=2,
    seed=SEED,
)

if GRADIENT_CHECKPOINTING:
    model.config.use_cache = False
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=tokenized['train'], eval_dataset=eval_gen,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=tokenized['train'], eval_dataset=eval_gen,
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        compute_metrics=compute_metrics,
    )

train_result = trainer.train()
metrics = trainer.evaluate()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

run_record = {
    'model_name': MODEL_NAME,
    'training_mode': 'kaggle',
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'mixed_precision': 'bf16' if bf16_ok else ('fp16' if torch.cuda.is_available() else 'no'),
    'train_examples': len(dataset['train']),
    'validation_examples': len(dataset['validation']),
    'eval_generation_examples': len(eval_gen),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'train_metrics': train_result.metrics,
    'validation_metrics': metrics,
}
(OUT_DIR / 'training_run.json').write_text(json.dumps(run_record, indent=2, default=str) + '\n', encoding='utf-8')
print(json.dumps(metrics, indent=2, default=str))

In [ ]:
# Zip checkpoint output
zip_path = shutil.make_archive(str(WORK / 'flan-t5-squadv2'), 'zip', root_dir=str(OUT_DIR))
print('Zipped checkpoint:', zip_path)
print()
print('Contents of the checkpoint folder:')
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print('  ', p.name, round(p.stat().st_size / 1e6, 1), 'MB')
print()
print('NEXT STEPS (local):')
print('  1. Download flan-t5-squadv2.zip from the Output panel.')
print('  2. Unzip into: models/question_generation/flan-t5-squadv2/')

## NOTE (post-mortem)
The previous run diverged (train_loss ~27.5, degenerate 'a a a' outputs) because T5 was fine-tuned in fp16 on a T4.
Fixed: model now loads in fp32 (or bf16 on Ampere+), and fp16 is never enabled. Re-run all cells to retrain.